# ai-arch: Phase 1 — End-to-end on ONE FSR doc

Goal: walk through `ingest → parse → chunk → embed → index → retrieve → generate` on a single FSR PDF, one stage at a time. Minimal code per step so we can review and discuss before adding complexity.

Sample doc: `ai-arch/sample-docs/fsr-sample-01.pdf` (~13 MB real FSR).

## Step 1 — Ingest

"Ingest" just means: get the file into a place our code can read it from. In prod, this would be a Unity Catalog **Volume** with Auto Loader picking up new files. For the POC, we point at the local copy in the repo so we can iterate fast.

In [2]:
from pathlib import Path

# Set ONE of these. Comment out the other.
# Local path (running locally in VS Code):
PDF_PATH = Path("/home/u560060992/dbx/ai-arch/sample-docs/fsr-sample-01.pdf")
# Databricks Repos path (when running on Databricks workspace):
# PDF_PATH = Path("/Workspace/Repos/<me>/dbx/ai-arch/sample-docs/fsr-sample-01.pdf")

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"
print(f"file: {PDF_PATH.name}")
print(f"size: {PDF_PATH.stat().st_size / 1024 / 1024:.2f} MB")

file: fsr-sample-01.pdf
size: 13.05 MB


## Step 2 — Parse (PDF → text)

Simplest possible parser first: **PyMuPDF** (`fitz`). It's fast, text-only, no OCR, no layout smarts. Good baseline — we'll see what it misses, then swap in better parsers in Phase 2.

In [1]:
%pip install -q pymupdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
import fitz  # PyMuPDF

doc = fitz.open(PDF_PATH)

print(f"pages: {doc.page_count}")
print(f"metadata: {doc.metadata}")

# Pull text page by page; keep page number alongside (we'll need it later for citations).
pages = [{"page": i + 1, "text": doc.load_page(i).get_text()} for i in range(doc.page_count)]
doc.close()

total_chars = sum(len(p["text"]) for p in pages)
print(f"total chars extracted: {total_chars:,}")
print(f"avg chars/page: {total_chars // len(pages):,}")

pages: 51
metadata: {'format': 'PDF 1.3', 'title': 'Unit 14 visual', 'author': '', 'subject': '', 'keywords': '', 'creator': '204073360', 'producer': 'pdfmake', 'creationDate': 'D:20251020155722Z', 'modDate': '', 'trapped': '', 'encryption': None}
total chars extracted: 20,243
avg chars/page: 396


### Eyeball the parse

Print the first ~500 chars of page 1 and a middle page. Look for:
- Is the text readable, or garbled (would suggest OCR is needed)?
- Did tables / form fields survive in any usable form?
- Headers / footers repeated on every page?

In [5]:
print("---- page 1 ----")
print(pages[0]["text"][:500])
print()
mid = len(pages) // 2
print(f"---- page {mid + 1} ----")
print(pages[mid]["text"][:500])

---- page 1 ----
 
Unit 14 visual
A Inspection 
MIDLAND COGEN
 
Outage Start Date: 20 Aug 2025
ESN/SY: 810893 | SY0048230
Oracle Project ID: A-1960052 | EV-185177 | EVP-555614
Report Issued: 20 Oct 2025
Prepared By
Justin Clark
Field Engineer ( TFA ) (FC)
Approved By
James Burgess
Customer Service Leader (GP)
© 2025, GE Vernova. GE Vernova Confidential Information - This document contains GE Vernova proprietary information. It is the property of GE Vernova 
and shall not be used, disclosed to others or reproduce

---- page 26 ----
2
.
3
.
3
 
H
G
C
 
F
l
o
w
 
S
t
r
u
t
 
S
e
p
e
r
a
t
o
r
2.3.3 HGC Flow Strut Seperator
F
O
R
M
Part Description: 
HGC flow strut separator was found in acceptable condition.
IMG 2053
GEGEGE
GE Vernova Confidential and Proprietary Information. Not to be copied, reproduced or distributed without GE Vernova prior written consent.
A Inspection - 810893
MCV Partners LLC
MIDLAND COGEN
Page 23 of 48

